In [2]:
import os
from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm
from google.adk.sessions import InMemorySessionService
from google.adk.runners import Runner
from google.genai import types 
from typing import Optional, Dict, Any

import warnings
warnings.filterwarnings("ignore")

import logging
logging.basicConfig(level=logging.CRITICAL)


import litellm
import logging

logging.getLogger("LiteLLM").setLevel(logging.CRITICAL)
logging.getLogger("litellm").setLevel(logging.CRITICAL)
logging.disable(logging.CRITICAL)  

In [3]:
MODEL_GPT = "groq/openai/gpt-oss-120b"
llm = LiteLlm(model=MODEL_GPT, reasoning_format="hidden")

print(
    llm.llm_client.completion(
        model=llm.model,
        messages=[
            {
                "role": "user",
                "content": "Are you ready?"
            }
        ],
        tools=[]
    )
)
print("\nGroq is ready for use.")

ModelResponse(id='chatcmpl-ce1f1ae7-11a2-4515-8d62-216c4d498c45', created=1789386144, model='openai/gpt-oss-120b', object='chat.completion', system_fingerprint='fp_fe269835c7', choices=[Choices(finish_reason='stop', index=0, message=Message(content='Absolutely—I’m ready! How can I help you today?', role='assistant', tool_calls=None, function_call=None, provider_specific_fields=None, reasoning='The user asks "Are you sure? Are you ready?" Actually they said "Are you ready?" It\'s a simple question. We can respond affirmatively.'))], usage=Usage(completion_tokens=53, prompt_tokens=75, total_tokens=128, completion_tokens_details=CompletionTokensDetailsWrapper(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=32, rejected_prediction_tokens=None, text_tokens=None, image_tokens=None, video_tokens=None), prompt_tokens_details=None, queue_time=0.349583066, prompt_time=0.004427631, completion_time=0.111715566, total_time=0.116143197), usage_breakdown=None, x_groq={'id': 'req_

In [4]:
from helper import make_agent_caller

In [5]:
def say_hello(person_name: str) -> dict:
    """Formats a welcome message to a named person. 

    Args:
        person_name (str): the name of the person saying hello

    Returns:
        dict: A dictionary containing the results of the query.
              Includes a 'status' key ('success' or 'error').
              If 'success', includes a 'query_result' key with an array of result rows.
              If 'error', includes an 'error_message' key.
    """
    return graphdb.send_query("RETURN 'Hello to you, ' + $person_name AS reply",
    {
        "person_name": person_name
    })

In [6]:
def say_goodbye() -> dict:
    """Provides a simple farewell message to conclude the conversation."""
    return graphdb.send_query("RETURN 'Goodbye from Cypher!' as farewell")

# Sub-Agents (Greeting & Farewell)

In [7]:
# Greeting Agent 
greeting_subagent = Agent(
    model=llm,
    name="greeting_subagent_v1",
    instruction="You are the Greeting Agent. Your ONLY task is to provide a friendly greeting to the user. "
                "Use the 'say_hello' tool to generate the greeting. "
                "If the user provides their name, make sure to pass it to the tool. "
                "Do not engage in any other conversation or tasks.",
    description="Handles simple greetings and hellos using the 'say_hello' tool.", 
    tools=[say_hello],
)
print(f"Agent '{greeting_subagent.name}' created.")


Agent 'greeting_subagent_v1' created.


In [8]:
# Farewell Agent 
farewell_subagent = Agent(
    model=llm, 
    name="farewell_subagent_v1",
    instruction="You are the Farewell Agent. Your ONLY task is to provide a polite goodbye message. "
                "Use the 'say_goodbye' tool when the user indicates they are leaving or ending the conversation "
                "(e.g., using words like 'bye', 'goodbye', 'thanks bye', 'see you'). "
                "Do not perform any other actions.",
    description="Handles simple farewells and goodbyes using the 'say_goodbye' tool.", 
    tools=[say_goodbye],
)
print(f"Agent '{farewell_subagent.name}' created.")

Agent 'farewell_subagent_v1' created.


# Root Agent with Sub-Agents

In [9]:
root_agent = Agent(
    name="friendly_agent_team_v1", 
    model=llm,
    description="The main coordinator agent. Delegates greetings/farewells to specialists.",
    instruction="""You are the main Agent coordinating a team. Your primary responsibility is to be friendly.
 
                You have specialized sub-agents: 
                1. 'greeting_agent': Handles simple greetings like 'Hi', 'Hello'. Delegate to it for these. 
                2. 'farewell_agent': Handles simple farewells like 'Bye', 'See you'. Delegate to it for these. 

                Analyze the user's query. If it's a greeting, delegate to 'greeting_agent'. 
                If it's a farewell, delegate to 'farewell_agent'. 
                
                For anything else, respond appropriately or state you cannot handle it.
                """,
    tools=[], 
    sub_agents=[greeting_subagent, farewell_subagent]
)


print(f"Root Agent '{root_agent.name}' created with sub-agents: {[sa.name for sa in root_agent.sub_agents]}")


Root Agent 'friendly_agent_team_v1' created with sub-agents: ['greeting_subagent_v1', 'farewell_subagent_v1']
